# 🤖 한국어 AI 텍스트 탐지기 — Colab 발표용 Dual 모델

이 Notebook은 **두 가지 모델을 한 입력기에서 비교**할 수 있도록 구성합니다.

- **기본 모델:** Word TF-IDF + Character TF-IDF + Logistic Regression
- **Kiwi 모델:** Word TF-IDF + Character TF-IDF + Kiwi 형태소 TF-IDF + Logistic Regression

마지막에는 Gradio 입력기를 실행하여 같은 텍스트를 두 모델에 동시에 넣고 결과를 비교할 수 있습니다.

> 먼저 기본 모델과 Kiwi 모델을 각각 학습한 뒤 마지막 Gradio 셀을 실행하세요.


## 1. Colab 환경 및 패키지

In [1]:
!pip install -q gradio kiwipiepy scikit-learn scipy pandas numpy joblib tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 33.8 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import json
import hashlib
import random
import re
import time
import numpy as np
import pandas as pd
import joblib

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix
)
from kiwipiepy import Kiwi

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "katfish"
MODEL_DIR = Path("models")

DATA_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

print("환경 준비 완료")


환경 준비 완료


In [6]:
# ==========================================
# 공개 데이터셋 자동 다운로드
# ==========================================

from pathlib import Path
import subprocess
import shutil
import os

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "katfish"

DATA_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

REPO_URL = "https://github.com/Shinwoo-Park/katfishnet.git"
REPO_DIR = Path("katfishnet")

if not REPO_DIR.exists():
    print("KatFishNet 저장소 다운로드 중...")
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True
    )
else:
    print("KatFishNet 저장소가 이미 존재합니다.")

print("\n저장소 내부 JSONL 파일 검색 중...")

jsonl_files = list(REPO_DIR.rglob("*.jsonl"))

for f in jsonl_files:
    print(f)

print(f"\n발견한 JSONL 파일: {len(jsonl_files)}개")

KatFishNet 저장소 다운로드 중...

저장소 내부 JSONL 파일 검색 중...
katfishnet/katfish_dataset/essay.jsonl
katfishnet/katfish_dataset/poetry.jsonl
katfishnet/katfish_dataset/abstract.jsonl

발견한 JSONL 파일: 3개


## 2. 데이터 로딩

실제 JSONL의 `label`을 직접 사용합니다.

- `0` = Human
- `1` = AI


In [7]:
# ==========================================
# KatFishNet 데이터셋 자동 준비 + 로딩
# ==========================================

from pathlib import Path
import json
import hashlib
import pandas as pd
import re

REPO_DIR = Path("katfishnet")
DATASET_DIR = REPO_DIR / "katfish_dataset"

jsonl_files = sorted(DATASET_DIR.glob("*.jsonl"))

if not jsonl_files:
    raise FileNotFoundError(
        "KatFishNet 데이터셋을 찾을 수 없습니다."
    )

print("===== KatFishNet Dataset =====")

for f in jsonl_files:
    print(
        f"{f.name}: "
        f"{f.stat().st_size / 1024 / 1024:.2f} MB"
    )

rows = []

for path in jsonl_files:

    print(f"\nLoading: {path.name}")

    with path.open(
        encoding="utf-8"
    ) as f:

        for line_no, line in enumerate(
            f,
            start=1
        ):

            if not line.strip():
                continue

            try:
                x = json.loads(line)

            except json.JSONDecodeError:
                print(
                    f"JSON 오류: "
                    f"{path.name}:{line_no}"
                )
                continue

            # ==================================
            # 실제 KatFish 데이터에서 text 찾기
            # ==================================

            text = None

            for key in [
                "text",
                "essay",
                "content",
                "sentence",
                "document",
                "abstract",
                "poem"
            ]:

                value = x.get(key)

                if (
                    isinstance(value, str)
                    and value.strip()
                ):
                    text = value.strip()
                    break

            # ==================================
            # label
            # ==================================

            label_value = x.get("label")

            try:
                label = int(label_value)

            except (
                TypeError,
                ValueError
            ):

                label_str = (
                    str(label_value)
                    .lower()
                    .strip()
                )

                if label_str in [
                    "human",
                    "0"
                ]:
                    label = 0

                elif label_str in [
                    "ai",
                    "generated",
                    "machine",
                    "1"
                ]:
                    label = 1

                else:
                    continue

            if label not in [0, 1]:
                continue

            if not text:
                continue

            # ==================================
            # 데이터 저장
            # ==================================

            rows.append({

                "id":
                    f"katfish_{len(rows):08d}",

                "text":
                    text,

                "label":
                    label,

                "source_type":
                    "human"
                    if label == 0
                    else "ai",

                "domain":
                    path.stem,

                "original_id":
                    hashlib.sha1(
                        text.encode("utf-8")
                    ).hexdigest()[:16]
            })


# ==========================================
# DataFrame
# ==========================================

if not rows:

    raise RuntimeError(
        "데이터를 하나도 읽지 못했습니다."
    )

df = pd.DataFrame(rows)


# ==========================================
# 텍스트 정규화
# ==========================================

df["text"] = (
    df["text"]
    .astype(str)
    .str.replace(
        "\u200b",
        " ",
        regex=False
    )
    .str.replace(
        "\ufeff",
        " ",
        regex=False
    )
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.strip()
)


# ==========================================
# 너무 짧은 텍스트 제거
# ==========================================

df = df[
    df["text"].str.len() >= 30
].copy()


# ==========================================
# 중복 제거
# ==========================================

df = df.drop_duplicates(
    subset=["text", "label"]
).reset_index(drop=True)


# ==========================================
# DATA CHECK
# ==========================================

print("\n")
print("=" * 45)
print("DATA CHECK")
print("=" * 45)

print(
    f"rows: {len(df)}"
)

print("\nlabels:")
print(
    df["label"]
    .value_counts()
    .sort_index()
)

print("\nsource_type:")
print(
    df["source_type"]
    .value_counts()
)

print("\ndomain:")
print(
    df["domain"]
    .value_counts()
)


# ==========================================
# 클래스 확인
# ==========================================

if df["label"].nunique() < 2:

    raise ValueError(
        "Human(0)과 AI(1) 두 클래스가 모두 필요합니다.\n"
        f"현재 데이터:\n"
        f"{df['label'].value_counts()}"
    )

print("\n✅ KatFishNet 데이터 로딩 성공!")

===== KatFishNet Dataset =====
abstract.jsonl: 4.94 MB
essay.jsonl: 2.71 MB
poetry.jsonl: 0.59 MB

Loading: abstract.jsonl

Loading: essay.jsonl

Loading: poetry.jsonl


DATA CHECK
rows: 2094

labels:
label
0     470
1    1624
Name: count, dtype: int64

source_type:
source_type
ai       1624
human     470
Name: count, dtype: int64

domain:
domain
poetry      945
essay       771
abstract    378
Name: count, dtype: int64

✅ KatFishNet 데이터 로딩 성공!


## 3. Train / Validation / Test 분할

In [8]:
train, temp = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["label"]
)

val, test = train_test_split(
    temp,
    test_size=0.50,
    random_state=SEED,
    stratify=temp["label"]
)

for name, part in [
    ("Train", train),
    ("Validation", val),
    ("Test", test)
]:
    print(f"\n{name}: {len(part)}")
    print(part["label"].value_counts().sort_index())

    if part["label"].nunique() < 2:
        raise ValueError(f"{name}에 두 클래스가 모두 없습니다.")



Train: 1675
label
0     376
1    1299
Name: count, dtype: int64

Validation: 209
label
0     47
1    162
Name: count, dtype: int64

Test: 210
label
0     47
1    163
Name: count, dtype: int64


# 4. 기본 모델 학습

Word + Character TF-IDF만 사용합니다.

CPU와 메모리 사용량을 줄이기 위해 feature 수를 제한합니다.


In [9]:
word_vec_base = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_features=30000,
    sublinear_tf=True
)

char_vec_base = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    max_features=40000,
    sublinear_tf=True
)

Xbw_train = word_vec_base.fit_transform(train["text"])
Xbw_val = word_vec_base.transform(val["text"])
Xbw_test = word_vec_base.transform(test["text"])

Xbc_train = char_vec_base.fit_transform(train["text"])
Xbc_val = char_vec_base.transform(val["text"])
Xbc_test = char_vec_base.transform(test["text"])

X_base_train = hstack(
    [Xbw_train, Xbc_train],
    format="csr"
)

X_base_val = hstack(
    [Xbw_val, Xbc_val],
    format="csr"
)

X_base_test = hstack(
    [Xbw_test, Xbc_test],
    format="csr"
)

print("기본 feature:", X_base_train.shape)


기본 feature: (1675, 70000)


In [10]:
base_clf = LogisticRegression(
    C=3.0,
    max_iter=300,
    class_weight="balanced",
    solver="liblinear",
    random_state=SEED
)

t0 = time.time()
base_clf.fit(X_base_train, train["label"])

print(f"기본 모델 학습 시간: {time.time()-t0:.2f}s")
print("classes:", base_clf.classes_)


기본 모델 학습 시간: 0.28s
classes: [0 1]


## 5. 기본 모델 평가

In [11]:
def evaluate_model(model, X, y, name):
    prob = model.predict_proba(X)[:, 1]
    pred = (prob >= 0.5).astype(int)

    result = {
        "model": name,
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "roc_auc": roc_auc_score(y, prob)
    }

    print(f"\n===== {name} / TEST =====")
    print(classification_report(
        y, pred,
        target_names=["Human", "AI"],
        zero_division=0
    ))
    print("Confusion matrix:")
    print(confusion_matrix(y, pred))

    return result

base_result = evaluate_model(
    base_clf,
    X_base_test,
    test["label"],
    "기본 모델"
)

print(base_result)



===== 기본 모델 / TEST =====
              precision    recall  f1-score   support

       Human       0.88      0.77      0.82        47
          AI       0.93      0.97      0.95       163

    accuracy                           0.92       210
   macro avg       0.91      0.87      0.88       210
weighted avg       0.92      0.92      0.92       210

Confusion matrix:
[[ 36  11]
 [  5 158]]
{'model': '기본 모델', 'accuracy': 0.9238095238095239, 'precision': 0.9349112426035503, 'recall': 0.9693251533742331, 'f1': 0.9518072289156626, 'roc_auc': np.float64(0.9664534656050124)}


# 6. Kiwi 적용 모델 학습

Kiwi 형태소 분석 결과를 별도의 TF-IDF feature로 만들어 기본 feature에 추가합니다.


In [12]:
kiwi = Kiwi()

def kiwi_texts(texts):
    result = []

    for text in texts:
        tokens = kiwi.tokenize(str(text))

        result.append(
            " ".join(
                f"{token.form}/{token.tag}"
                for token in tokens
            )
        )

    return result

print("Train Kiwi 분석 중...")
kt_train = kiwi_texts(train["text"].tolist())

print("Validation Kiwi 분석 중...")
kt_val = kiwi_texts(val["text"].tolist())

print("Test Kiwi 분석 중...")
kt_test = kiwi_texts(test["text"].tolist())

print("Kiwi 분석 완료")


Train Kiwi 분석 중...
Validation Kiwi 분석 중...
Test Kiwi 분석 중...
Kiwi 분석 완료


In [13]:
pos_vec_kiwi = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_features=20000,
    sublinear_tf=True
)

Xp_train = pos_vec_kiwi.fit_transform(kt_train)
Xp_val = pos_vec_kiwi.transform(kt_val)
Xp_test = pos_vec_kiwi.transform(kt_test)

X_kiwi_train = hstack(
    [X_base_train, Xp_train],
    format="csr"
)

X_kiwi_val = hstack(
    [X_base_val, Xp_val],
    format="csr"
)

X_kiwi_test = hstack(
    [X_base_test, Xp_test],
    format="csr"
)

print("Kiwi feature:", Xp_train.shape)
print("전체 Kiwi 모델 feature:", X_kiwi_train.shape)


Kiwi feature: (1675, 20000)
전체 Kiwi 모델 feature: (1675, 90000)


In [14]:
kiwi_clf = LogisticRegression(
    C=3.0,
    max_iter=300,
    class_weight="balanced",
    solver="liblinear",
    random_state=SEED
)

t0 = time.time()
kiwi_clf.fit(X_kiwi_train, train["label"])

print(f"Kiwi 모델 학습 시간: {time.time()-t0:.2f}s")
print("classes:", kiwi_clf.classes_)


Kiwi 모델 학습 시간: 0.58s
classes: [0 1]


## 7. Kiwi 모델 평가 및 두 모델 비교

In [15]:
kiwi_result = evaluate_model(
    kiwi_clf,
    X_kiwi_test,
    test["label"],
    "Kiwi 모델"
)

comparison = pd.DataFrame([
    base_result,
    kiwi_result
])

display(comparison)



===== Kiwi 모델 / TEST =====
              precision    recall  f1-score   support

       Human       0.84      0.79      0.81        47
          AI       0.94      0.96      0.95       163

    accuracy                           0.92       210
   macro avg       0.89      0.87      0.88       210
weighted avg       0.92      0.92      0.92       210

Confusion matrix:
[[ 37  10]
 [  7 156]]


,model,accuracy,precision,recall,f1,roc_auc
0,기본 모델,0.923810,0.934911,0.969325,0.951807,0.966453
1,Kiwi 모델,0.919048,0.939759,0.957055,0.948328,0.967889


# 8. 두 모델 저장

Gradio 입력기는 이 두 모델을 현재 Colab 메모리에서 바로 사용합니다.

또한 나중에 재사용할 수 있도록 `.joblib` 파일로 저장합니다.


In [17]:
# ==========================================
# Cell 16 — 모델 저장
# ==========================================

import joblib
from pathlib import Path

MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

# -----------------------------
# 기본 모델
# -----------------------------

base_bundle = {
    "word_vectorizer": word_vec_base,
    "char_vectorizer": char_vec_base,
    "classifier": base_clf,
    "threshold": 0.5,
    "model_type": "base"
}

# -----------------------------
# Kiwi 모델
# -----------------------------
# 중요:
# Kiwi 객체 자체는 pickle/joblib으로 저장하지 않는다.
# 실제 사용 시 Kiwi()를 새로 생성한다.

kiwi_bundle = {
    "word_vectorizer": word_vec_base,
    "char_vectorizer": char_vec_base,
    "pos_vectorizer": pos_vec_kiwi,
    "classifier": kiwi_clf,
    "threshold": 0.5,
    "model_type": "kiwi"
}

base_path = MODEL_DIR / "ko_ai_detector_base.joblib"
kiwi_path = MODEL_DIR / "ko_ai_detector_kiwi.joblib"

joblib.dump(
    base_bundle,
    base_path,
    compress=3
)

joblib.dump(
    kiwi_bundle,
    kiwi_path,
    compress=3
)

# -----------------------------
# 평가 결과 저장
# -----------------------------

comparison.to_csv(
    DATA_DIR / "model_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("================================")
print("MODEL SAVE COMPLETE")
print("================================")

print("기본 모델:")
print(base_path)

print("\nKiwi 모델:")
print(kiwi_path)

print("\n비교 결과:")
print(DATA_DIR / "model_comparison.csv")

MODEL SAVE COMPLETE
기본 모델:
models/ko_ai_detector_base.joblib

Kiwi 모델:
models/ko_ai_detector_kiwi.joblib

비교 결과:
data/model_comparison.csv


# 9. 🎨 Gradio 비교 입력기

같은 문장을 두 모델에 동시에 넣습니다.

화면에는:

- 기본 모델 AI 확률
- Kiwi 모델 AI 확률
- 두 모델의 판정
- 두 모델의 차이

가 표시됩니다.


In [18]:
import gradio as gr

def normalize_app(text):
    text = str(text)
    text = text.replace("\u200b", " ")
    text = text.replace("\ufeff", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def make_base_features_app(text):
    text = normalize_app(text)

    Xw = word_vec_base.transform([text])
    Xc = char_vec_base.transform([text])

    return hstack([Xw, Xc], format="csr")


def make_kiwi_features_app(text):
    text = normalize_app(text)

    Xw = word_vec_base.transform([text])
    Xc = char_vec_base.transform([text])

    tokens = kiwi.tokenize(text)

    kiwi_text = " ".join(
        f"{token.form}/{token.tag}"
        for token in tokens
    )

    Xp = pos_vec_kiwi.transform([kiwi_text])

    return hstack(
        [Xw, Xc, Xp],
        format="csr"
    )


def predict_both(text):
    if not text or not text.strip():
        return (
            "텍스트를 입력해주세요.",
            0.0,
            "텍스트를 입력해주세요.",
            0.0,
            "입력 대기 중"
        )

    text = normalize_app(text)

    if len(text) < 30:
        return (
            "30자 이상의 텍스트를 입력해주세요.",
            0.0,
            "30자 이상의 텍스트를 입력해주세요.",
            0.0,
            "텍스트가 너무 짧습니다."
        )

    # -------------------------
    # 기본 모델
    # -------------------------
    X_base = make_base_features_app(text)

    base_prob = float(
        base_clf.predict_proba(X_base)[0, 1]
    )

    # -------------------------
    # Kiwi 모델
    # -------------------------
    X_kiwi = make_kiwi_features_app(text)

    kiwi_prob = float(
        kiwi_clf.predict_proba(X_kiwi)[0, 1]
    )

    base_pct = base_prob * 100
    kiwi_pct = kiwi_prob * 100

    base_result = (
        "🤖 AI 생성 가능성이 높음"
        if base_prob >= 0.5
        else "👤 Human 작성 가능성이 높음"
    )

    kiwi_result = (
        "🤖 AI 생성 가능성이 높음"
        if kiwi_prob >= 0.5
        else "👤 Human 작성 가능성이 높음"
    )

    difference = abs(base_pct - kiwi_pct)

    summary = (
        f"기본 모델: {base_pct:.1f}%\n"
        f"Kiwi 모델: {kiwi_pct:.1f}%\n"
        f"두 모델 차이: {difference:.1f}%p"
    )

    return (
        base_result,
        round(base_pct, 2),
        kiwi_result,
        round(kiwi_pct, 2),
        summary
    )


demo = gr.Interface(
    fn=predict_both,

    inputs=gr.Textbox(
        lines=12,
        label="분석할 한국어 텍스트",
        placeholder=(
            "AI 생성 여부를 확인할 텍스트를 입력하세요..."
        )
    ),

    outputs=[
        gr.Textbox(
            label="기본 모델 판정"
        ),
        gr.Number(
            label="기본 모델 AI 가능성 (%)"
        ),
        gr.Textbox(
            label="Kiwi 모델 판정"
        ),
        gr.Number(
            label="Kiwi 모델 AI 가능성 (%)"
        ),
        gr.Textbox(
            label="모델 비교"
        )
    ],

    title="🤖 한국어 AI 텍스트 탐지기",

    description=(
        "기본 TF-IDF 모델과 Kiwi 형태소 분석 모델을 "
        "동시에 비교합니다.\n\n"
        "⚠️ 결과는 머신러닝 모델의 예측값이며 "
        "실제 작성자를 확정적으로 판별하는 것은 아닙니다."
    ),

    examples=[
        [
            "오늘 학교에 조금 일찍 도착해서 도서관에서 책을 읽었다. "
            "평소보다 시간이 많아서 관심이 있던 책을 천천히 읽을 수 있었다."
        ],
        [
            "효율적인 학습을 위해서는 명확한 목표를 설정하고 "
            "체계적인 계획을 수립하는 것이 중요하다. "
            "이를 통해 학습자는 자신의 진행 상황을 지속적으로 점검할 수 있다."
        ]
    ]
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1a4f42e6e14f4ec7a9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# 10. 발표용 사용 방법

1. Colab에서 **1번부터 9번까지 순서대로 실행**
2. 마지막 Gradio 셀에서 `Running on public URL` 확인
3. 생성된 `gradio.live` 링크 클릭
4. 텍스트 입력
5. **기본 모델과 Kiwi 모델의 결과를 동시에 비교**

### 발표에서 설명할 수 있는 구조

**기본 모델**
> Word/Character TF-IDF → Logistic Regression

**Kiwi 모델**
> Word/Character TF-IDF + Kiwi 형태소 TF-IDF → Logistic Regression

두 모델의 결과 차이를 비교함으로써 **형태소 정보가 AI 탐지 성능에 어떤 영향을 주는지** 확인할 수 있습니다.

> ⚠️ AI 생성 확률은 모델의 예측값이며, 실제 작성자를 확정하는 증거가 아닙니다.
